# Сеть плана: обучение на Colab

Вход - 8 каналов 2 см (`simscan/netinput.py`), выход - классы (улица, помещение, стена, дверь, окно,
проём, мусор, пустота), расстояние до грани, ориентация стены, высоты проёмов.

Перед запуском: **Среда выполнения → Сменить среду → GPU** (T4 хватает, A100/L4 быстрее).

На Google Drive должны лежать:
* `MyDrive/plannet/dataset.zip` - датасет (`python -m simscan make-dataset ...` и `python -m simscan dataset pack dataset --out dataset.zip` на машине с ядрами);
* `MyDrive/plannet/real/*.e57` - реальные объекты для проверки (по желанию).

Прогон пишется в `MyDrive/plannet/runs/<имя>`: если сессия оборвётся, перезапустите все ячейки -
обучение продолжится с `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/plannet'
RUN = f'{DRIVE}/runs/r1'          # имя прогона
EPOCHS, BATCH, CROP = 60, 8, 512  # T4: BATCH 8 при CROP 512; не хватает памяти - BATCH 4

In [ ]:
!git clone -q -b claude/serene-einstein-9avsce https://github.com/BykovVadimCV/DotCloudCONVERT /content/DotCloudCONVERT 2>/dev/null || git -C /content/DotCloudCONVERT pull -q
%cd /content/DotCloudCONVERT
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# датасет - на локальный диск Colab (с Drive читать медленно)
import os
if not os.path.exists('/content/dataset/meta.json'):
    !unzip -q -o {DRIVE}/dataset.zip -d /content/
!ls /content/dataset; wc -l /content/dataset/splits/*.txt

In [ ]:
# обучение; --resume продолжает с last.pt, если он уже есть в RUN
!python -m simscan train --data /content/dataset --out {RUN} --epochs {EPOCHS} --batch {BATCH} --crop {CROP} --workers 2 --resume

In [ ]:
# кривые обучения
import json, matplotlib.pyplot as plt
h = [json.loads(l) for l in open(f'{RUN}/history.jsonl')]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r['total'] for r in h], label='train'); ax[0].plot([r.get('val_total') for r in h], label='val'); ax[0].set_title('потеря'); ax[0].legend()
ax[1].plot([r.get('miou_walls') for r in h]); ax[1].set_title('val mIoU: стена, дверь, окно, проём')
plt.show()
names = ['улица', 'помещение', 'стена', 'дверь', 'окно', 'проём', 'мусор', 'пустота']
print({n: v for n, v in zip(names, h[-1].get('iou', []))})

In [ ]:
# применение к реальным объектам: pred/<файл>/L<k>/pred.png, sem.png, prob.npz ...
import glob
real = sorted(glob.glob(f'{DRIVE}/real/*.e57'))
if real:
    !python -m simscan infer {RUN}/best.pt {' '.join(real)} --out {RUN}/pred

In [ ]:
from IPython.display import Image, display
for p in sorted(glob.glob(f'{RUN}/pred/*/L*/pred.png')):
    print(p); display(Image(p, width=900))